# Binary Classification for High-Dimensional Single-Cell Data

This notebook demonstrates binary classification approaches for high-dimensional single-cell RNA sequencing and spatial transcriptomics data. We'll explore different classification scenarios relevant to cancer research and spatial biology.

## Overview

Binary classification tasks in single-cell genomics can include:
1. **Cell type classification**: Distinguishing between specific cell types (e.g., malignant vs non-malignant)
2. **Spatial region classification**: Identifying tumor vs normal tissue regions
3. **Treatment response prediction**: Predicting response to therapy
4. **Integration quality assessment**: Identifying well-integrated vs poorly-integrated cells

We'll demonstrate these approaches using the integrated scRNA-seq and MERFISH datasets from this repository.

In [ ]:
# Import required packages
import sys
import os
import numpy as np
import pandas as pd
import scanpy as sc
import anndata as ad
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

# Import our binary classification module
from binary_classification import HighDimBinaryClassifier, create_synthetic_classification_data, run_classification_example

# Set up scanpy settings
sc.settings.verbosity = 1
sc.settings.set_figure_params(dpi=80, facecolor='white')

# Set random seed for reproducibility
np.random.seed(42)

print("Packages loaded successfully!")

## 1. Load and Prepare Data

Let's start by creating a synthetic dataset to demonstrate our binary classification approaches. In practice, you would load your actual integrated scRNA-seq/MERFISH data here.

In [ ]:
# Create a synthetic high-dimensional dataset for demonstration
# In practice, you would load your actual data here:
# adata = sc.read_h5ad('path_to_your_integrated_data.h5ad')

def create_demo_dataset(n_cells=2000, n_genes=5000):
    """Create a synthetic dataset for demonstration purposes."""
    print(f"Creating synthetic dataset with {n_cells} cells and {n_genes} genes...")
    
    # Generate synthetic expression data
    np.random.seed(42)
    X = np.random.negative_binomial(5, 0.3, size=(n_cells, n_genes)).astype(float)
    
    # Add some structure to make classification meaningful
    # Create two cell populations with different expression patterns
    n_diff_genes = n_genes // 10  # 10% of genes are differentially expressed
    
    # Population 1 (first half of cells) - upregulate first set of genes
    X[:n_cells//2, :n_diff_genes] *= 3
    
    # Population 2 (second half of cells) - upregulate second set of genes  
    X[n_cells//2:, n_diff_genes:2*n_diff_genes] *= 3
    
    # Create AnnData object
    adata = ad.AnnData(X)
    adata.var_names = [f'Gene_{i:04d}' for i in range(n_genes)]
    adata.obs_names = [f'Cell_{i:04d}' for i in range(n_cells)]
    
    # Add metadata
    adata.obs['cell_population'] = ['Pop_A' if i < n_cells//2 else 'Pop_B' for i in range(n_cells)]
    
    # Add some spatial coordinates for spatial analysis
    adata.obs['x_coordinate'] = np.random.uniform(0, 100, n_cells)
    adata.obs['y_coordinate'] = np.random.uniform(0, 100, n_cells)
    
    # Create tumor vs normal regions based on spatial coordinates
    # Define a circular tumor region in the center
    center_x, center_y = 50, 50
    tumor_radius = 25
    distances = np.sqrt((adata.obs['x_coordinate'] - center_x)**2 + 
                       (adata.obs['y_coordinate'] - center_y)**2)
    adata.obs['tissue_region'] = ['Tumor' if d < tumor_radius else 'Normal' for d in distances]
    
    # Add some cell type annotations
    cell_types = ['T_cell', 'B_cell', 'Macrophage', 'Epithelial', 'Fibroblast']
    adata.obs['cell_type'] = np.random.choice(cell_types, n_cells)
    
    # Create malignant vs non-malignant classification
    # Make epithelial cells in tumor region more likely to be malignant
    malignant_prob = np.where(
        (adata.obs['cell_type'] == 'Epithelial') & (adata.obs['tissue_region'] == 'Tumor'),
        0.8, 0.1
    )
    adata.obs['malignant_status'] = np.random.binomial(1, malignant_prob, n_cells)
    adata.obs['malignant_status'] = adata.obs['malignant_status'].map({1: 'Malignant', 0: 'Benign'})
    
    print(f"Dataset created successfully!")
    print(f"Cell population distribution: {adata.obs['cell_population'].value_counts().to_dict()}")
    print(f"Tissue region distribution: {adata.obs['tissue_region'].value_counts().to_dict()}")
    print(f"Malignant status distribution: {adata.obs['malignant_status'].value_counts().to_dict()}")
    
    return adata

# Create demonstration dataset
adata = create_demo_dataset(n_cells=2000, n_genes=3000)

# Basic preprocessing
sc.pp.normalize_total(adata, target_sum=1e4)
sc.pp.log1p(adata)
sc.pp.highly_variable_genes(adata, n_top_genes=1000)

print(f"\nDataset shape: {adata.shape}")
print(f"Highly variable genes: {adata.var.highly_variable.sum()}")

## 2. Exploratory Data Analysis

Let's visualize our data to understand the structure and potential classification targets.

In [ ]:
# Create visualizations of the data
fig, axes = plt.subplots(2, 2, figsize=(12, 10))

# Plot 1: Spatial distribution colored by tissue region
ax1 = axes[0, 0]
scatter = ax1.scatter(adata.obs['x_coordinate'], adata.obs['y_coordinate'], 
                     c=adata.obs['tissue_region'].map({'Tumor': 'red', 'Normal': 'blue'}),
                     alpha=0.6, s=20)
ax1.set_xlabel('X coordinate')
ax1.set_ylabel('Y coordinate')
ax1.set_title('Spatial Distribution by Tissue Region')
ax1.legend(['Normal', 'Tumor'])

# Plot 2: Spatial distribution colored by malignant status
ax2 = axes[0, 1]
scatter = ax2.scatter(adata.obs['x_coordinate'], adata.obs['y_coordinate'], 
                     c=adata.obs['malignant_status'].map({'Malignant': 'orange', 'Benign': 'green'}),
                     alpha=0.6, s=20)
ax2.set_xlabel('X coordinate')
ax2.set_ylabel('Y coordinate')
ax2.set_title('Spatial Distribution by Malignant Status')
ax2.legend(['Benign', 'Malignant'])

# Plot 3: Cell type distribution
ax3 = axes[1, 0]
cell_type_counts = adata.obs['cell_type'].value_counts()
ax3.bar(cell_type_counts.index, cell_type_counts.values)
ax3.set_xlabel('Cell Type')
ax3.set_ylabel('Count')
ax3.set_title('Cell Type Distribution')
ax3.tick_params(axis='x', rotation=45)

# Plot 4: Cross-tabulation of tissue region vs malignant status
ax4 = axes[1, 1]
crosstab = pd.crosstab(adata.obs['tissue_region'], adata.obs['malignant_status'])
sns.heatmap(crosstab, annot=True, fmt='d', cmap='Blues', ax=ax4)
ax4.set_title('Tissue Region vs Malignant Status')

plt.tight_layout()
plt.show()

# Print summary statistics
print("\n=== Dataset Summary ===")
print(f"Total cells: {adata.n_obs}")
print(f"Total genes: {adata.n_vars}")
print(f"\nCell type distribution:")
print(adata.obs['cell_type'].value_counts())
print(f"\nTissue region distribution:")
print(adata.obs['tissue_region'].value_counts())
print(f"\nMalignant status distribution:")
print(adata.obs['malignant_status'].value_counts())

## 3. Binary Classification Examples

Now let's demonstrate different binary classification scenarios using our high-dimensional data.

### 3.1 Malignant vs Benign Cell Classification

This is a clinically relevant classification task where we want to distinguish malignant from benign cells based on their gene expression profiles.

In [ ]:
print("=== Malignant vs Benign Cell Classification ===")

# Initialize classifier
classifier_malignant = HighDimBinaryClassifier(random_state=42)

# Prepare data - use highly variable genes for better performance
adata_hv = adata[:, adata.var.highly_variable].copy()
X, y = classifier_malignant.prepare_data(
    adata_hv, 
    target_column='malignant_status',
    positive_class='Malignant',
    negative_class='Benign'
)

# Feature selection - select top 200 features
X_selected = classifier_malignant.select_features(X, y, method='selectk', n_features=200)

# Split data for final evaluation
X_train, X_test, y_train, y_test = train_test_split(X_selected, y, test_size=0.2, stratify=y, random_state=42)

# Train model
classifier_malignant.train_model(X_train, y_train, model_type='random_forest', n_estimators=200)

# Evaluate with cross-validation on training set
cv_metrics = classifier_malignant.evaluate_model(X_train, y_train, cv_folds=5)

# Final evaluation on test set
if hasattr(classifier_malignant.scaler, 'mean_'):
    X_test_scaled = classifier_malignant.scaler.transform(X_test)
else:
    X_test_scaled = X_test

y_pred = classifier_malignant.model.predict(X_test_scaled)
print("\n=== Test Set Performance ===")
print(classification_report(y_test, y_pred, target_names=['Benign', 'Malignant']))

# Plot results
classifier_malignant.plot_results(X_test, y_test, figsize=(15, 5))

### 3.2 Tumor vs Normal Tissue Region Classification

This classification task focuses on spatial regions, distinguishing tumor areas from normal tissue areas.

In [ ]:
print("=== Tumor vs Normal Tissue Region Classification ===")

# Initialize classifier
classifier_region = HighDimBinaryClassifier(random_state=42)

# Prepare data
X, y = classifier_region.prepare_data(
    adata_hv, 
    target_column='tissue_region',
    positive_class='Tumor',
    negative_class='Normal'
)

# Try PCA for dimensionality reduction
X_selected = classifier_region.select_features(X, y, method='pca', n_features=100)

# Split data
X_train, X_test, y_train, y_test = train_test_split(X_selected, y, test_size=0.2, stratify=y, random_state=42)

# Train logistic regression model
classifier_region.train_model(X_train, y_train, model_type='logistic', C=1.0)

# Evaluate
cv_metrics = classifier_region.evaluate_model(X_train, y_train, cv_folds=5)

# Test set evaluation
if hasattr(classifier_region.scaler, 'mean_'):
    X_test_scaled = classifier_region.scaler.transform(X_test)
else:
    X_test_scaled = X_test

y_pred = classifier_region.model.predict(X_test_scaled)
print("\n=== Test Set Performance ===")
print(classification_report(y_test, y_pred, target_names=['Normal', 'Tumor']))

# Plot results
classifier_region.plot_results(X_test, y_test, show_feature_importance=False, figsize=(10, 5))

### 3.3 Cell Population Classification

This demonstrates classification between two distinct cell populations based on their expression profiles.

In [ ]:
print("=== Cell Population A vs B Classification ===")

# Initialize classifier
classifier_population = HighDimBinaryClassifier(random_state=42)

# Prepare data
X, y = classifier_population.prepare_data(
    adata_hv, 
    target_column='cell_population',
    positive_class='Pop_A',
    negative_class='Pop_B'
)

# Use RFE for feature selection
X_selected = classifier_population.select_features(X, y, method='rfe', n_features=150)

# Split data
X_train, X_test, y_train, y_test = train_test_split(X_selected, y, test_size=0.2, stratify=y, random_state=42)

# Train SVM model
classifier_population.train_model(X_train, y_train, model_type='svm', C=1.0, kernel='rbf')

# Evaluate
cv_metrics = classifier_population.evaluate_model(X_train, y_train, cv_folds=5)

# Test set evaluation
if hasattr(classifier_population.scaler, 'mean_'):
    X_test_scaled = classifier_population.scaler.transform(X_test)
else:
    X_test_scaled = X_test

y_pred = classifier_population.model.predict(X_test_scaled)
print("\n=== Test Set Performance ===")
print(classification_report(y_test, y_pred, target_names=['Pop_B', 'Pop_A']))

# Plot results
classifier_population.plot_results(X_test, y_test, figsize=(15, 5))

## 4. Advanced Analysis: Feature Interpretation

Let's examine which features (genes) are most important for our classifications.

In [ ]:
print("=== Feature Importance Analysis ===")

# Get feature importance for malignant vs benign classification
malignant_features = classifier_malignant.get_feature_importance(top_n=20)
print("\nTop features for Malignant vs Benign classification:")
print(malignant_features)

# Get feature importance for cell population classification
population_features = classifier_population.get_feature_importance(top_n=20)
print("\nTop features for Cell Population classification:")
print(population_features)

# Create a comparison plot
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Plot 1: Malignant vs Benign features
ax1 = axes[0]
sns.barplot(data=malignant_features.head(15), x='importance', y='feature', ax=ax1, palette='viridis')
ax1.set_title('Top Features: Malignant vs Benign Classification')
ax1.set_xlabel('Feature Importance')

# Plot 2: Population features
ax2 = axes[1]
sns.barplot(data=population_features.head(15), x='importance', y='feature', ax=ax2, palette='plasma')
ax2.set_title('Top Features: Cell Population Classification')
ax2.set_xlabel('Feature Importance')

plt.tight_layout()
plt.show()

## 5. Model Comparison

Let's compare different algorithms for the same classification task.

In [ ]:
print("=== Model Comparison for Malignant vs Benign Classification ===")

# Prepare data once
classifier_comp = HighDimBinaryClassifier(random_state=42)
X, y = classifier_comp.prepare_data(
    adata_hv, 
    target_column='malignant_status',
    positive_class='Malignant',
    negative_class='Benign'
)
X_selected = classifier_comp.select_features(X, y, method='selectk', n_features=200)

# Test different models
models_to_test = [
    ('Random Forest', 'random_forest', {'n_estimators': 100}),
    ('Logistic Regression', 'logistic', {'C': 1.0}),
    ('SVM', 'svm', {'C': 1.0, 'kernel': 'rbf'})
]

results = []

for model_name, model_type, params in models_to_test:
    print(f"\nTesting {model_name}...")
    
    # Initialize new classifier
    clf = HighDimBinaryClassifier(random_state=42)
    
    # Train model
    clf.train_model(X_selected, y, model_type=model_type, **params)
    
    # Evaluate
    metrics = clf.evaluate_model(X_selected, y, cv_folds=5)
    
    results.append({
        'Model': model_name,
        'Accuracy': f"{metrics['accuracy_mean']:.3f} ± {metrics['accuracy_std']:.3f}",
        'Precision': f"{metrics['precision_mean']:.3f} ± {metrics['precision_std']:.3f}",
        'Recall': f"{metrics['recall_mean']:.3f} ± {metrics['recall_std']:.3f}",
        'F1': f"{metrics['f1_mean']:.3f} ± {metrics['f1_std']:.3f}",
        'ROC-AUC': f"{metrics['roc_auc_mean']:.3f} ± {metrics['roc_auc_std']:.3f}"
    })

# Display results
results_df = pd.DataFrame(results)
print("\n=== Model Comparison Results ===")
print(results_df.to_string(index=False))

## 6. Using Real Data (Optional)

If you have actual integrated scRNA-seq/MERFISH data, you can use it here. Uncomment and modify the code below to load your data.

In [ ]:
# # Load your actual integrated data
# # Example for loading data that might exist in this repository
# 
# try:
#     # Try to load any existing integrated data
#     # Adjust paths based on actual data files in the repository
#     
#     # Example paths - modify based on actual data location
#     data_paths = [
#         '../data/integrated_data.h5ad',
#         '../integration/integrated_output.h5ad',
#         # Add other potential paths here
#     ]
#     
#     for path in data_paths:
#         if os.path.exists(path):
#             print(f"Loading data from {path}")
#             real_adata = sc.read_h5ad(path)
#             print(f"Loaded real data: {real_adata.shape}")
#             print(f"Available annotations: {list(real_adata.obs.columns)}")
#             
#             # Run classification on real data
#             # You would specify appropriate target columns and classes based on your data
#             # classifier_real = run_classification_example(
#             #     real_adata, 
#             #     target_column='your_target_column',
#             #     positive_class='your_positive_class',
#             #     negative_class='your_negative_class'
#             # )
#             break
#     else:
#         print("No real data files found in expected locations")
#         
# except Exception as e:
#     print(f"Could not load real data: {e}")
#     print("Using synthetic data for demonstration")

print("\nTo use real data:")
print("1. Ensure your integrated data is saved as an AnnData object (.h5ad file)")
print("2. Update the data paths in the code above")
print("3. Specify appropriate target columns and class labels")
print("4. Run the classification analysis")

## 7. Summary and Next Steps

This notebook demonstrated several approaches to binary classification for high-dimensional single-cell data:

### Key Findings:
1. **Feature Selection**: Different methods (SelectKBest, RFE, PCA) can be used depending on the data characteristics
2. **Model Performance**: Random Forest, Logistic Regression, and SVM all showed good performance
3. **Biological Relevance**: The classification tasks demonstrated are relevant to cancer research

### Best Practices:
- Use cross-validation for robust performance estimation
- Scale features when using distance-based algorithms
- Select features appropriate for high-dimensional data
- Interpret feature importance in biological context

### Next Steps:
1. Apply these methods to your actual integrated scRNA-seq/MERFISH data
2. Explore ensemble methods for improved performance
3. Incorporate spatial information for spatial transcriptomics data
4. Validate findings with independent datasets
5. Investigate biological pathways enriched in important features

In [ ]:
print("Binary classification analysis completed!")
print("\nModule capabilities:")
print("- High-dimensional feature selection")
print("- Multiple classification algorithms")
print("- Cross-validation and robust evaluation")
print("- Feature importance analysis")
print("- Visualization tools")
print("\nReady for application to real single-cell and spatial transcriptomics data!")